# Walmart 周销售额预测

这份运行记录对应同目录的研究报告和 `walmart_analysis.py`。先检查数据，再看特征处理、训练曲线、测试误差和权重。默认读取已经训练得到的结果，重新训练只需修改下面的开关。

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image
from walmart_analysis import main, make_features, scores
RETRAIN = False
DATA = Path('Walmart_Sales.csv')
OUT = Path('results')
if RETRAIN:
    main(DATA, OUT)
summary = json.loads((OUT/'summary.json').read_text(encoding='utf-8'))


## 1. 读入与检查

日期使用日-月-年格式。门店编号和日期共同标识一条记录，不能把日期当作一般字符串输入模型。

In [2]:
df = pd.read_csv(DATA)
print('shape:', df.shape)
print('missing:', df.isna().sum().to_dict())
print('duplicate Store-Date:', df.duplicated(['Store','Date']).sum())
display(df.head())

shape: (6435, 8)
missing: {'Store': 0, 'Date': 0, 'Weekly_Sales': 0, 'Holiday_Flag': 0, 'Temperature': 0, 'Fuel_Price': 0, 'CPI': 0, 'Unemployment': 0}
duplicate Store-Date: 0


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


## 2. 特征与数据划分

按周排序取前约60%训练，中间约20%验证，后约20%测试；同一周的全部门店放在同一集合。数值特征的均值、标准差和独热编码都仅从训练集拟合。

In [3]:
df['Date'] = pd.to_datetime(df.Date, format='%d-%m-%Y')
df = df.sort_values(['Date','Store']).reset_index(drop=True)
df['Year'], df['Month'] = df.Date.dt.year, df.Date.dt.month
df['Week'] = df.Date.dt.isocalendar().week.astype(int)
df['Week_sin'] = np.sin(2*np.pi*df.Week/52.18)
df['Week_cos'] = np.cos(2*np.pi*df.Week/52.18)
dates = np.sort(df.Date.unique())
a, b = int(len(dates)*.6), int(len(dates)*.8)
train = df.Date.isin(dates[:a]).to_numpy()
test = df.Date.isin(dates[b:]).to_numpy()
x, feature_names, scaler, encoder = make_features(df, train, summary['selected_model'])
print('selected model:', summary['selected_model'])
print('feature count excluding intercept:', x.shape[1]-1)
display(pd.DataFrame(summary['split']).T)

selected model: store_month_encoded
feature count excluding intercept: 63


,rows,weeks,start,end
train,3825,85,2010-02-05,2011-09-16
validation,1305,29,2011-09-23,2012-04-06
test,1305,29,2012-04-13,2012-10-26


## 3. 模型训练

课堂里的线性预测可写成 `prediction = X @ w + b`。配套脚本使用均方误差的梯度下降训练，验证损失决定权重保存轮次，验证集决定特征方案。下表只比较预先设置的三种线性模型。

In [4]:
rows=[]
for name, data in summary['models'].items():
    rows.append({'Model': name, 'Validation_RMSE': data['validation']['RMSE'], 'Validation_R2': data['validation']['R2'], 'Selected_Update': data['best_epoch']})
display(pd.DataFrame(rows))
display(Image(filename=str(OUT/'figures/03_loss.png')))

,Model,Validation_RMSE,Validation_R2,Selected_Update
0,numeric_store,558702.554441,0.146785,11
1,store_encoded,220288.489877,0.867358,8000
2,store_month_encoded,190659.033067,0.900640,8000


<IPython.core.display.Image object>

## 4. 对保存的权重重新计算测试预测

下面没有重新调参，直接用保存的权重算预测，再和结果表核对。

In [5]:
model = np.load(OUT/'linear_model.npz')
pred = x @ model['weights'] * model['y_scale'] + model['y_mean']
saved = pd.read_csv(OUT/'predictions.csv')
assert np.allclose(pred, saved.Predicted_Sales.to_numpy())
print('test metrics:', scores(df.Weekly_Sales.to_numpy()[test], pred[test]))
print('store mean baseline:', summary['store_mean_baseline_test'])
display(Image(filename=str(OUT/'figures/04_actual_prediction.png')))

test metrics: {'MAE': 87030.31799793038, 'MSE': 13399280589.935043, 'RMSE': 115755.26160799363, 'R2': 0.9529372995342885, 'WAPE': 0.08387068488540907}
store mean baseline: {'MAE': 74628.687626865, 'MSE': 11754959816.263975, 'RMSE': 108420.29245608949, 'R2': 0.9587126973641509, 'WAPE': 0.07191929533697977}


<IPython.core.display.Image object>

## 5. 分布、相关性、残差与特征权重

In [6]:
for name in ['01_distributions','02_correlations','07_holiday_store','05_residuals','06_weights']:
    display(Image(filename=str(OUT/'figures'/f'{name}.png')))

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

<IPython.core.display.Image object>

## 6. 这次结果说明什么

门店独热编码比直接输入门店编号有效。最后模型R²为0.9529，但门店历史均值基线的测试MAE更小。因此，较高的总体R²主要说明模型能区分门店销售规模，不能直接证明它更适合预测每家门店下一周的波动。后续应增加只使用过去观测的销售滞后特征，并使用滚动时间验证。

温度、油价、CPI、失业率在本次测试中取数据表记录值，因此结果属于给定这些变量后的条件预测。真正提前预测时，还需要这些变量当时可获得的估计值。